## Exploratory Data Analysis (EDA)

Este notebook realiza el análisis exploratorio del corpus canónico generado por `ingest.py`.

**Prerrequisito:** haber ejecutado `python src/ingest.py` para generar `data/corpus_canonical_v1.jsonl`

Pasos que cubre este notebook:
- Resumen estadístico básico del corpus
- Extracción y frecuencia de hashtags
- Detección de multimedia (imágenes y vídeos)
- Visualizaciones básicas

In [ ]:
# Instalacion de Librerias
#%pip install missingno

In [ ]:
#1 Importacion de Librerias

import pandas as pd               # Manipulación y análisis de datos; DataFrame, lectura/escritura CSV/JSON
import numpy as np                # Operaciones numéricas y arrays; utilidades estadísticas y matemáticas
import matplotlib.pyplot as plt   # Visualización básica; crear figuras, ejes y guardar gráficos
import seaborn as sns             # Visualizaciones estadísticas de alto nivel; estilos y gráficos complejos
import re                         # Librería para trabajar con expresiones regulares

In [ ]:
# =======================================================================
# 1. Cargar corpus canónico generado por ingest.py
# Este archivo es el punto de partida del EDA.
# Debe existir antes de ejecutar este notebook.

corpus_path = "data/corpus_canonical_v1.jsonl"

df_es = pd.read_json(corpus_path, lines=True)

print(f"✅ Corpus cargado correctamente: {len(df_es)} tweets en español")

In [ ]:
# =======================================================================
# 2. Vista preliminar de los datos
# Revisamos las primeras filas para entender la estructura del corpus canónico.

print("\n=== Vista preliminar del corpus ===")
print(df_es.head())

In [ ]:
# =======================================================================
# 3. Dimensiones y tipos de datos
# shape devuelve número de filas y columnas.
# dtypes muestra el tipo de dato de cada columna (numérico, texto, etc.).

print("\n=== Dimensiones y tipos de datos ===")
print("Dimensiones:", df_es.shape)
print("\nTipos de datos:\n", df_es.dtypes)

In [ ]:
# =======================================================================
# 4. Estadísticas básicas
# describe() genera estadísticas descriptivas de columnas numéricas
# y con include="all" también incluye categóricas.

print("\n=== Estadísticas básicas ===")
print(df_es.describe(include="all"))

# RESUMEN ESTADÍSTICO BÁSICO

In [ ]:
# 5. =======================================================================
# RESUMEN ESTADISTICO BASICO

print("\n=== RESUMEN ESTADISTICO BASICO ===\n")

# Columnas del corpus canónico
print("\n=== Columnas del corpus canónico ===\n")
print(df_es.columns.tolist())

# Número total de tweets
print("\nTotal de tweets en español:", len(df_es))

# Distribución por idioma (aunque ya filtramos, sirve para verificar)
print("\n=== Distribución por idioma ===")
print(df_es['lang'].value_counts())

# Longitud promedio de los tweets (usar 'full_text')
df_es['tweet_length'] = df_es['full_text'].apply(lambda x: len(str(x)))
print("\n=== Longitud promedio de los tweets ===")
print(df_es['tweet_length'].mean())


# 5.1. =======================================================================
# EXTRACCIÓN DE HASHTAGS Y MULTIMEDIA DESDE EL TEXTO

# Extraer hashtags directamente del texto de los tweets usando regex
# La expresión r"#\w+" busca cualquier palabra precedida por el símbolo #
df_es['extracted_hashtags'] = df_es['full_text'].apply(
    lambda x: re.findall(r"#\w+", str(x))
)

# Unimos todas las listas de hashtags en una sola lista
all_hashtags = df_es['extracted_hashtags'].sum()

# Convertimos la lista en una Serie de pandas para poder contar frecuencias
hashtags_series = pd.Series(all_hashtags)

# Mostramos los 10 hashtags más frecuentes en el corpus
print("\n=== Top 10 hashtags más frecuentes ===")
print(hashtags_series.value_counts().head(10))

# Detectar enlaces a imágenes en el texto
# 'pic.twitter.com' es el patrón típico de imágenes compartidas en tweets
df_es['has_image'] = df_es['full_text'].apply(lambda x: 'pic.twitter.com' in str(x))

# Detectar enlaces a videos en el texto
# Buscamos palabras clave como 'video' o enlaces a YouTube ('youtu')
df_es['has_video'] = df_es['full_text'].apply(
    lambda x: 'video' in str(x).lower() or 'youtu' in str(x).lower()
)

# Mostramos el conteo de tweets que contienen multimedia
print("\n=== Tweets con multimedia detectada ===")
print("Con imagen:", df_es['has_image'].sum())
print("Con video:", df_es['has_video'].sum())


"""
# Hashtags más frecuentes (solo si existe la columna 'hashtags')
if 'hashtags' in df_es.columns:
    try:
        # Convertir listas de hashtags en una sola lista
        all_hashtags = df_es['hashtags'].dropna().apply(eval).sum()
        hashtags_series = pd.Series(all_hashtags)
        print("\n=== Top 10 hashtags más frecuentes ===")
        print(hashtags_series.value_counts().head(10))
    except Exception as e:
        print("Error procesando hashtags:", e)

# Tweets con imágenes y videos (solo si existen esas columnas)
if 'is_image' in df_es.columns and 'is_video' in df_es.columns:
    print("\n=== Tweets con multimedia ===")
    print("Con imagen:", df_es['is_image'].astype(int).sum())
    print("Con video:", df_es['is_video'].astype(int).sum())
"""

In [ ]:
# 6. =======================================================================
# VISUALIZACIONES BÁSICAS

# VISUALIZACIÓN COMBINADA: LONGITUDES DE TWEETS + HASHTAGS FRECUENTES

# --- Preparar datos para hashtags ---
# Unimos todas las listas de hashtags en una sola lista
all_hashtags = df_es['extracted_hashtags'].sum()

# Convertimos la lista en una Serie de pandas para contar frecuencias
hashtags_series = pd.Series(all_hashtags)

# Seleccionamos los 10 hashtags más frecuentes
top_hashtags = hashtags_series.value_counts().head(10)

# --- Crear figura con dos subplots ---
plt.figure(figsize=(14,6))  # Definimos tamaño general de la figura

# Subplot 1: Histograma de longitudes de tweets
plt.subplot(1,2,1)  # 1 fila, 2 columnas, posición 1
plt.hist(df_es['tweet_length'], bins=30, color='skyblue', edgecolor='black')
plt.title("Distribución de la longitud de los tweets")
plt.xlabel("Número de caracteres")
plt.ylabel("Frecuencia")

# Subplot 2: Gráfico de barras de hashtags más frecuentes
plt.subplot(1,2,2)  # 1 fila, 2 columnas, posición 2
top_hashtags.plot(kind='bar', color='salmon', edgecolor='black')
plt.title("Top 10 hashtags más frecuentes")
plt.xlabel("Hashtag")
plt.ylabel("Frecuencia")
plt.xticks(rotation=45)

# Ajustamos el diseño para que no se solapen los gráficos
plt.tight_layout()

# Mostramos la figura completa
plt.show()